# Backtest min_count_not_met ????

????????
- ?????????????qcut ???? bins?
- ????? min_count_not_met


In [3]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path(r"C:/Users/BaiYang/CBOND_ON")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cbond_on.core.config import load_config_file, parse_date
from cbond_on.data.io import read_table_range, read_trading_calendar
from cbond_on.models.score_io import load_scores_by_date
from cbond_on.core.universe import filter_tradable

paths_cfg = load_config_file("paths")
backtest_cfg = load_config_file("backtest")
live_cfg = load_config_file("live")
ms_cfg = backtest_cfg.get("model_score", {})
model_cfg = load_config_file(str(ms_cfg.get("model_config", "models/linear/model")))

raw_root = paths_cfg["raw_data_root"]
start = parse_date(backtest_cfg["start"])
end = parse_date(backtest_cfg["end"])
score_path = Path(model_cfg["score_output"])
scores = load_scores_by_date(str(score_path)) if score_path.exists() else {}

buy_twap_col = backtest_cfg["buy_twap_col"]
sell_twap_col = backtest_cfg["sell_twap_col"]
min_count = int(backtest_cfg["min_count"])
ic_bins = int(backtest_cfg.get("ic_bins", 20))
bin_top_k = max(1, int(backtest_cfg.get("bin_top_k", 1)))

filter_flag = bool(live_cfg.get("filter_tradable", True))
min_amount = float(live_cfg.get("min_amount", 0))
min_volume = float(live_cfg.get("min_volume", 0))

def _read_twap_daily(raw_data_root, day):
    df = read_table_range(raw_data_root, "market_cbond.daily_twap", day, day)
    if df.empty:
        return df
    if "instrument_code" in df.columns and "exchange_code" in df.columns:
        df = df.copy()
        df["code"] = df["instrument_code"].astype(str) + "." + df["exchange_code"].astype(str)
    return df

def _trading_days(raw_data_root):
    cal = read_trading_calendar(raw_data_root)
    if cal.empty:
        return []
    c = cal.copy()
    c["calendar_date"] = pd.to_datetime(c["calendar_date"], errors="coerce").dt.date
    if "is_open" in c.columns:
        c = c[c["is_open"].astype(bool)]
    return sorted(c["calendar_date"].dropna().unique().tolist())

day_list = [d for d in _trading_days(raw_root) if start <= d <= end]
print("score_path:", score_path)
print("days:", len(day_list))


score_path: D:\cbond_on\results\scores\lgbm_factor_default\scores.csv
days: 32


In [4]:
rows = []
for i in range(len(day_list) - 1):
    day = day_list[i]
    next_day = day_list[i + 1]
    rec = {"trade_date": day, "next_day": next_day}

    score_df = scores.get(day, pd.DataFrame())
    rec["score_count"] = int(len(score_df))
    if score_df.empty:
        rec["reason"] = "missing_score"
        rec["bins_actual"] = 0
        rows.append(rec)
        continue

    buy_df = _read_twap_daily(raw_root, day)
    sell_df = _read_twap_daily(raw_root, next_day)
    if buy_df.empty or sell_df.empty:
        rec["reason"] = "missing_clean"
        rec["bins_actual"] = 0
        rows.append(rec)
        continue

    merged = buy_df.merge(
        sell_df[["code", sell_twap_col]],
        on="code",
        how="left",
        suffixes=("", "_next"),
    )
    if sell_twap_col not in merged.columns and f"{sell_twap_col}_next" in merged.columns:
        merged[sell_twap_col] = merged[f"{sell_twap_col}_next"]
    merged = merged.merge(score_df[["code", "score"]], on="code", how="left")

    if filter_flag:
        merged = filter_tradable(
            merged,
            buy_twap_col=buy_twap_col,
            sell_twap_col=sell_twap_col,
            min_amount=min_amount,
            min_volume=min_volume,
        )

    merged = merged[merged["score"].notna()].copy()
    rec["tradable_with_score"] = int(len(merged))
    if merged.empty:
        rec["reason"] = "no_tradable"
        rec["bins_actual"] = 0
        rows.append(rec)
        continue

    try:
        bins_cat = pd.qcut(merged["score"], ic_bins, labels=False, duplicates="drop")
    except Exception:
        bins_cat = None

    if bins_cat is None or bins_cat.dropna().empty:
        rec["reason"] = "binning_failed"
        rec["bins_actual"] = 0
        rows.append(rec)
        continue

    merged["bin"] = bins_cat.astype(float)
    merged = merged[merged["bin"].notna()].copy()
    merged["bin"] = merged["bin"].astype(int)
    available_bins = sorted(merged["bin"].unique().tolist())

    rec["bins_actual"] = int(len(available_bins))
    rec["bins_expected"] = int(ic_bins)
    rec["bins_collapsed"] = bool(len(available_bins) < ic_bins)

    chosen_bins = sorted(available_bins, reverse=True)[:bin_top_k]
    picks = merged[merged["bin"].isin(chosen_bins)]
    rec["chosen_bins"] = ",".join(map(str, chosen_bins))
    rec["picks_count"] = int(len(picks))
    rec["min_count"] = int(min_count)

    if len(picks) < min_count:
        rec["reason"] = "min_count_not_met"
    else:
        rec["reason"] = "ok"

    rows.append(rec)

diag = pd.DataFrame(rows).sort_values("trade_date").reset_index(drop=True)
diag.head()


,trade_date,next_day,score_count,reason,bins_actual,tradable_with_score,bins_expected,bins_collapsed,chosen_bins,picks_count,min_count
0,2026-01-05,2026-01-06,0,missing_score,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-01-06,2026-01-07,368,ok,18,368.0,20.0,True,"17,16",37.0,10.0
2,2026-01-07,2026-01-08,380,ok,20,380.0,20.0,False,"19,18",38.0,10.0
3,2026-01-08,2026-01-09,380,ok,20,380.0,20.0,False,"19,18",38.0,10.0
4,2026-01-09,2026-01-12,379,ok,20,379.0,20.0,False,"19,18",38.0,10.0


In [ ]:
print("=== ?????? ===")
display(diag[["trade_date", "bins_actual", "bins_expected", "bins_collapsed", "reason"]])

print("=== min_count_not_met ???? ===")
mc = diag[diag["reason"] == "min_count_not_met"].copy()
if mc.empty:
    print("?? min_count_not_met")
else:
    display(mc[[
        "trade_date",
        "score_count",
        "tradable_with_score",
        "bins_actual",
        "bins_collapsed",
        "chosen_bins",
        "picks_count",
        "min_count",
        "reason",
    ]])

out = ROOT / "results" / "backtest" / "min_count_diag_simple.csv"
out.parent.mkdir(parents=True, exist_ok=True)
diag.to_csv(out, index=False)
print("saved:", out)
diag

=== ?????? ===


,trade_date,bins_actual,bins_expected,bins_collapsed,reason
0,2026-01-05,0,NaN,NaN,missing_score
1,2026-01-06,18,20.0,True,ok
2,2026-01-07,20,20.0,False,ok
3,2026-01-08,20,20.0,False,ok
4,2026-01-09,20,20.0,False,ok
5,2026-01-12,0,NaN,NaN,missing_score
6,2026-01-13,20,20.0,False,ok
7,2026-01-14,20,20.0,False,ok
8,2026-01-15,20,20.0,False,ok
9,2026-01-16,20,20.0,False,ok


=== min_count_not_met ???? ===
?? min_count_not_met
saved: C:\Users\BaiYang\CBOND_ON\results\backtest\min_count_diag_simple.csv
